In [15]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

In [16]:
train_df=pd.read_csv("/kaggle/input/datasets/poushalisanyal/ml-merged/merged_train.csv")
test_df=pd.read_csv("/kaggle/input/datasets/poushalisanyal/ml-merged/merged_test.csv")
val_df=pd.read_csv("/kaggle/input/datasets/poushalisanyal/ml-merged/merged_valid.csv")

In [17]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

In [18]:
X_train = vectorizer.fit_transform(train_df['text'])
X_val = vectorizer.transform(val_df['text'])
X_test = vectorizer.transform(test_df['text'])

In [19]:
y_train = train_df['label']
y_val = val_df['label']
y_test = test_df['label']

In [20]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "SVM": LinearSVC(class_weight='balanced'),
    "Naive Bayes": MultinomialNB()
}

In [21]:
results = []
best_model = None
best_f1 = 0

for name, model in models.items():
    print("\n==============================")
    print(f"MODEL: {name}")
    print("==============================")

    # -------- TRAIN --------
    model.fit(X_train, y_train)

    # -------- VALIDATION --------
    y_val_pred = model.predict(X_val)
    val_acc = accuracy_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')
    val_precision = precision_score( y_val, y_val_pred, average='weighted' ) 
    val_recall = recall_score( y_val, y_val_pred, average='weighted' )

    print("\n--- Validation Performance ---")
    print("Accuracy:", val_acc)
    print("F1 Score:", val_f1)
    print("Precision:", val_precision) 
    print("Recall :", val_recall)

    # -------- TEST --------
    y_test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')

    print("\n--- Test Performance ---")
    print("Accuracy:", test_acc)
    print("F1 Score:", test_f1)

    print("\nClassification Report (Test):")
    print(classification_report(y_test, y_test_pred))

    # -------- STORE RESULTS --------
    results.append({
        "Model": name,
        "Val Accuracy": val_acc,
        "Val F1": val_f1,
        "Test Accuracy": test_acc,
        "Test F1": test_f1
    })

    # -------- BEST MODEL SELECTION --------
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model = name


MODEL: Logistic Regression

--- Validation Performance ---
Accuracy: 0.6003705991352687
F1 Score: 0.6087722415565199
Precision: 0.6499680411588038
Recall : 0.6003705991352687

--- Test Performance ---
Accuracy: 0.49793266391021856
F1 Score: 0.48936287337818996

Classification Report (Test):
              precision    recall  f1-score   support

    negative       0.55      0.46      0.50      2914
     neutral       0.41      0.74      0.53      2561
    positive       0.70      0.33      0.44      2990

    accuracy                           0.50      8465
   macro avg       0.55      0.51      0.49      8465
weighted avg       0.56      0.50      0.49      8465


MODEL: SVM

--- Validation Performance ---
Accuracy: 0.6046942557134033
F1 Score: 0.6105266896160594
Precision: 0.6257746434522359
Recall : 0.6046942557134033

--- Test Performance ---
Accuracy: 0.5108092144122859
F1 Score: 0.5093442475348084

Classification Report (Test):
              precision    recall  f1-score   suppo

In [22]:
results_df = pd.DataFrame(results)

print("\n\n==============================")
print("FINAL COMPARISON TABLE")
print("==============================")
print(results_df)

# ==========================================
# 6. BEST MODEL
# ==========================================
print("\nBest Model based on Validation F1 Score:", best_model)



FINAL COMPARISON TABLE
                 Model  Val Accuracy    Val F1  Test Accuracy   Test F1
0  Logistic Regression      0.600371  0.608772       0.497933  0.489363
1                  SVM      0.604694  0.610527       0.510809  0.509344
2          Naive Bayes      0.534898  0.543492       0.427171  0.396530

Best Model based on Validation F1 Score: SVM
